In [1]:
import os
import time
import sys
from pathlib import Path

import json
import uuid

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import col, concat_ws, current_timestamp, lit, sha2
from pyspark.sql.types import (
    LongType,
    StringType,
    StructField,
    StructType,
)

from etl.transformations.common import (
    create_spark,
    read_clickhouse,
    read_current_snapshot,
    write_clickhouse,
    epoch_to_timestamp,
)

from etl.transformations.dimensions.scd.common import (
    build_expired_version,
    build_new_version,
    add_hash,
    split_changes,
)

MINIO_STAGING_BUCKET = os.environ.get(
    "MINIO_STAGING_BUCKET",
    "staging",
)

In [2]:
def transform_dim_sensor(
    sensor_df: DataFrame,
    farm_df: DataFrame,
    sensor_type_df: DataFrame,
) -> DataFrame:
    """
    Build dim_sensor source shape.

    Converts source foreign keys:
        farm_id        -> farm_key
        sensor_type_id -> sensor_type_key

    using current warehouse dimension versions.
    """

    return (
        sensor_df.join(
            farm_df,
            sensor_df.farm_id == farm_df.farm_id,
            "left",
        )
        .join(
            sensor_type_df,
            sensor_df.sensor_type_id == sensor_type_df.sensor_type_id,
            "left",
        )
        .select(
            sensor_df.id.alias("sensor_id"),
            farm_df.farm_key,
            sensor_type_df.sensor_type_key,
            sensor_df.serial_number,
            sensor_df.status,
            epoch_to_timestamp(sensor_df.installed_at).alias("installed_at"),
        )
    )

In [3]:
def main():
    """
    Load dim_sensor SCD2 dimension.
    """

    spark = create_spark(
        "load_dim_sensor",
    )

    try:
        #
        # Read current snapshots from staging.
        #
        sensor_df = read_current_snapshot(
            spark,
            MINIO_STAGING_BUCKET,
            "sensors",
            primary_key="id",
        )

        #
        # Read current warehouse dimensions.
        #
        current_dim_farm_df = read_clickhouse(
            spark,
            """
            (
                SELECT *
                FROM dim_farm FINAL
            ) AS dim_farm
            """,
        ).filter(
            col("is_current") == 1,
        )

        current_dim_sensor_type_df = read_clickhouse(
            spark,
            """
            (
                SELECT *
                FROM dim_sensor_type FINAL
            ) AS dim_sensor_type
            """,
        ).filter(
            col("is_current") == 1,
        )

        #
        # Prepare current dimension lookup tables.
        #
        farm_lookup_df = current_dim_farm_df.select(
            "farm_id",
            "farm_key",
        )

        sensor_type_lookup_df = current_dim_sensor_type_df.select(
            "sensor_type_id",
            "sensor_type_key",
        )

        #
        # Create source shape for dim_sensor.
        #
        source_sensor_df = transform_dim_sensor(
            sensor_df,
            farm_lookup_df,
            sensor_type_lookup_df,
        )

        #
        # Read current SCD2 versions.
        #
        current_dim_sensor_df = read_clickhouse(
            spark,
            """
            (
                SELECT *
                FROM dim_sensor FINAL
            ) AS dim_sensor
            """,
        ).filter(
            col("is_current") == 1,
        )

        #
        # Detect new and changed sensors.
        #
        source_hashed_df = add_hash(
            source_sensor_df,
            [
                "farm_key",
                "sensor_type_key",
                "serial_number",
                "status",
                "installed_at",
            ],
        )

        current_hashed_df = add_hash(
            current_dim_sensor_df,
            [
                "farm_key",
                "sensor_type_key",
                "serial_number",
                "status",
                "installed_at",
            ],
        )

        new_sensors_df, changed_sensors_df = split_changes(
            source_hashed_df,
            current_hashed_df,
            "sensor_id",
        )

        #
        # Get current versions that must expire.
        #
        expired_sensors_df = current_dim_sensor_df.join(
            changed_sensors_df.select(
                "sensor_id",
            ),
            "sensor_id",
            "inner",
        )

        load_version = int(
            time.time() * 1000,
        )

        build_new_version(
                new_sensors_df,
                load_version,
                ["sensor_key"],
            ).printSchema()
        
        build_new_version(
                new_sensors_df,
                load_version,
                ["sensor_key"],
            ).show()

        build_expired_version(
                    expired_sensors_df,
                    load_version,
                     ["sensor_key"],
                ).printSchema()

        build_expired_version(
                    expired_sensors_df,
                    load_version,
                     ["sensor_key"],
                ).show()

        #
        # Build SCD2 rows.
        #
        rows_to_write = (
            build_new_version(
                new_sensors_df,
                load_version,
                ["sensor_key"],
            )
            .unionByName(
                build_expired_version(
                    expired_sensors_df,
                    load_version,
                    ["sensor_key"],
                )
            )
            .unionByName(
                build_new_version(
                    changed_sensors_df,
                    load_version,
                    ["sensor_key"],
                )
            )
        )

        #
        # Write warehouse changes.
        #
        write_clickhouse(
            rows_to_write,
            "dim_sensor",
        )

    finally:
        spark.stop()

In [4]:
if __name__ == "__main__":
    main()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/22 15:27:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/22 15:27:52 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


root
 |-- sensor_id: long (nullable = true)
 |-- farm_key: decimal(20,0) (nullable = true)
 |-- sensor_type_key: decimal(20,0) (nullable = true)
 |-- serial_number: string (nullable = true)
 |-- status: string (nullable = true)
 |-- installed_at: timestamp (nullable = true)
 |-- valid_from: timestamp (nullable = false)
 |-- valid_to: timestamp (nullable = true)
 |-- is_current: integer (nullable = false)
 |-- _version: long (nullable = false)

+---------+--------------------+--------------------+------------------+------+-------------------+--------------------+-------------------+----------+-------------+
|sensor_id|            farm_key|     sensor_type_key|     serial_number|status|       installed_at|          valid_from|           valid_to|is_current|     _version|
+---------+--------------------+--------------------+------------------+------+-------------------+--------------------+-------------------+----------+-------------+
|        1|13046267870273741923| 6312672892918141670| 

26/07/22 15:27:58 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
